# Portfolio Visualization

Visualizes portfolio progression from the analysis database.

**Prerequisites**: Run `maintain_portfolio.py` first to update databases

## Setup

In [1]:
import sys
from pathlib import Path
from datetime import datetime
import sqlite3
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Database path (in portfolio_tracker/data)
script_dir = Path.cwd()
db_path = script_dir / "data" / "portfolio_analysis.db"

print(f"Database: {db_path}\n")

if not db_path.exists():
    print("✗ Database not found!")
    print(f"Expected: {db_path}")
    print("Please run: python maintain_portfolio.py")
    sys.exit(1)

print("✓ Database found\n")

Database: c:\SynologyDrive\Personal\Beleggen\ETF Portfolio Management\Core Satellite\portfolio_tracker\data\portfolio_analysis.db

✓ Database found



## Load Portfolio Data

In [2]:
conn = sqlite3.connect(str(db_path))

# Get all dates with holdings
dates = pd.read_sql_query(
    "SELECT DISTINCT h.date FROM holdings h ORDER BY h.date",
    conn
)['date'].tolist()

# Load all transactions with quantity and buy/sell flag
trades_df = pd.read_sql_query(
    "SELECT etf_id, trade_date, price, quantity FROM trades ORDER BY trade_date",
    conn
)
trades_df['trade_date'] = pd.to_datetime(trades_df['trade_date'])
# Quantity is positive for buys, negative for sells in DEGIRO
# Calculate investment per trade: quantity × price (with sign)
trades_df['investment'] = trades_df['quantity'] * trades_df['price']

# Load cash balances and create a proper mapping
cash_df = pd.read_sql_query(
    "SELECT date, uninvested_cash FROM cash_balance ORDER BY date",
    conn
)
cash_map = {}
if len(cash_df) > 0:
    # Make sure dates are strings for matching with holdings dates
    for _, row in cash_df.iterrows():
        cash_map[row['date']] = row['uninvested_cash']

print(f"DEBUG: Loaded {len(cash_map)} cash entries")
if len(cash_map) > 0:
    print(f"Sample cash entries: {list(cash_map.items())[:3]}")

# Calculate portfolio value for each date
portfolio_values = []

for date_str in dates:
    # Get all holdings on this date
    holdings_df = pd.read_sql_query(
        """
        SELECT e.id, e.isin, e.name, h.quantity
        FROM holdings h
        JOIN etfs e ON h.etf_id = e.id
        WHERE h.date = ? AND h.quantity > 0
        """,
        conn,
        params=(date_str,)
    )

    # Calculate portfolio value (ETFs only)
    portfolio_value = 0
    current_date = pd.to_datetime(date_str)

    for _, holding in holdings_df.iterrows():
        etf_id = holding['id']
        
        # Always use market close price from prices table
        # This applies consistently for all dates, including buy/sell days
        price_query = pd.read_sql_query(
            "SELECT close_price FROM prices WHERE etf_id = ? AND date = ?",
            conn,
            params=(etf_id, date_str)
        )
        
        if len(price_query) > 0 and pd.notna(price_query['close_price'].iloc[0]):
            price = price_query['close_price'].iloc[0]
        else:
            # If no price for this date, skip (shouldn't happen with forward-fill)
            continue
        
        position_value = holding['quantity'] * price
        portfolio_value += position_value

    # Get uninvested cash for this date (use string key to match holdings date)
    uninvested_cash = cash_map.get(date_str, 0)

    # Total portfolio value = ETF positions + uninvested cash
    total_value = portfolio_value + uninvested_cash

    # Get cumulative contributions up to this date (sum of all investments)
    cumulative_invested = trades_df[trades_df['trade_date'] <= current_date]['investment'].sum()

    # Get cumulative fees
    fees_result = pd.read_sql_query(
        "SELECT SUM(fee) as total FROM trades WHERE trade_date <= ?",
        conn,
        params=(date_str,)
    )
    total_fees = abs(fees_result['total'].iloc[0]) if pd.notna(fees_result['total'].iloc[0]) else 0

    portfolio_values.append({
        'date': pd.to_datetime(date_str),
        'etf_value': portfolio_value,
        'uninvested_cash': uninvested_cash,
        'total_value': total_value,
        'cumulative_invested': cumulative_invested,
        'cumulative_fees': total_fees
    })

portfolio_df = pd.DataFrame(portfolio_values)
conn.close()

print(f"Loaded {len(portfolio_df)} days of portfolio data")

DEBUG: Loaded 38 cash entries
Sample cash entries: [('2026-01-02', 71.0), ('2026-01-03', 71.0), ('2026-01-04', 71.0)]
Loaded 38 days of portfolio data


## Portfolio Statistics

In [3]:
if len(portfolio_df) > 0:
    print("\n" + "="*80)
    print("PORTFOLIO STATISTICS")
    print("="*80 + "\n")

    print(f"Date range: {portfolio_df['date'].min().date()} to {portfolio_df['date'].max().date()}")
    print(f"Total days tracked: {len(portfolio_df)}")

    print(f"\nCurrent Status:")
    print(f"  ETF Value: €{portfolio_df['etf_value'].iloc[-1]:,.2f}")
    print(f"  Uninvested Cash: €{portfolio_df['uninvested_cash'].iloc[-1]:,.2f}")
    print(f"  Total Portfolio Value: €{portfolio_df['total_value'].iloc[-1]:,.2f}")
    print(f"  Cumulative Invested (spent on buys): €{portfolio_df['cumulative_invested'].iloc[-1]:,.2f}")
    print(f"  Fees Paid: €{portfolio_df['cumulative_fees'].iloc[-1]:,.2f}")
    
    # Unrealized gains = Current ETF Value - Amount Spent
    # Fees are already deducted from our purchasing power
    unrealized_gains = portfolio_df['etf_value'].iloc[-1] - portfolio_df['cumulative_invested'].iloc[-1]
    print(f"  Unrealized Gains (before fees): €{unrealized_gains:,.2f}")
    
    # Net gain after fees
    net_gain = unrealized_gains - portfolio_df['cumulative_fees'].iloc[-1]
    print(f"  Net Gain (after fees): €{net_gain:,.2f}")

    print(f"\nHistorical:")
    print(f"  Highest value: €{portfolio_df['total_value'].max():,.2f}")
    print(f"  Lowest value: €{portfolio_df['total_value'].min():,.2f}")

    print(f"\n" + "="*80 + "\n")
else:
    print("✗ No portfolio data found")


PORTFOLIO STATISTICS

Date range: 2026-01-02 to 2026-02-08
Total days tracked: 38

Current Status:
  ETF Value: €33,225.94
  Uninvested Cash: €71.00
  Total Portfolio Value: €33,296.94
  Cumulative Invested (spent on buys): €32,928.00
  Fees Paid: €1.00
  Unrealized Gains (before fees): €297.94
  Net Gain (after fees): €296.94

Historical:
  Highest value: €33,754.99
  Lowest value: €32,779.12




In [4]:
print("=" * 100)
print("DETAILED PORTFOLIO DATA - First 10 days")
print("=" * 100)
print(portfolio_df[['date', 'etf_value', 'uninvested_cash', 'total_value']].head(10).to_string())

print("\n" + "=" * 100)
print("DETAILED PORTFOLIO DATA - Last 5 days")
print("=" * 100)
print(portfolio_df[['date', 'etf_value', 'uninvested_cash', 'total_value']].tail(5).to_string())

print("\n" + "=" * 100)
print("PROBLEMATIC ROWS - Where total_value is outside normal range")
print("=" * 100)
# Find rows where total_value seems wrong
suspicious = portfolio_df[(portfolio_df['total_value'] > 35000) | (portfolio_df['total_value'] < 32000)]
print(suspicious[['date', 'etf_value', 'uninvested_cash', 'total_value']].to_string())
print(f"\nFound {len(suspicious)} suspicious rows")

DETAILED PORTFOLIO DATA - First 10 days
        date   etf_value  uninvested_cash  total_value
0 2026-01-02  32859.2628             71.0   32930.2628
1 2026-01-03  32859.2628             71.0   32930.2628
2 2026-01-04  32859.2628             71.0   32930.2628
3 2026-01-05  33091.2876             71.0   33162.2876
4 2026-01-06  33393.1080             71.0   33464.1080
5 2026-01-07  33261.2196             71.0   33332.2196
6 2026-01-08  33331.0152             71.0   33402.0152
7 2026-01-09  33623.6040             71.0   33694.6040
8 2026-01-10  33623.6040             71.0   33694.6040
9 2026-01-11  33623.6040             71.0   33694.6040

DETAILED PORTFOLIO DATA - Last 5 days
         date   etf_value  uninvested_cash  total_value
33 2026-02-04  33069.7080             71.0   33140.7080
34 2026-02-05  32708.1174             71.0   32779.1174
35 2026-02-06  33225.9396             71.0   33296.9396
36 2026-02-07  33225.9396             71.0   33296.9396
37 2026-02-08  33225.9396           

## Debug: View Detailed Data

## Portfolio Visualization

In [6]:
if len(portfolio_df) > 0:
    fig = go.Figure()

    # Total portfolio value (ETFs + cash)
    fig.add_trace(
        go.Scatter(
            x=portfolio_df['date'],
            y=portfolio_df['total_value'],
            name='Portfolio Value (ETFs + Cash)',
            mode='lines',
            line=dict(color='#00D9FF', width=2),
            fill='tozeroy',
            fillcolor='rgba(0, 217, 255, 0.2)',
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Value: €%{y:,.2f}<extra></extra>'
        )
    )

    # Cumulative invested amount
    fig.add_trace(
        go.Scatter(
            x=portfolio_df['date'],
            y=portfolio_df['cumulative_invested'],
            name='Cumulative Invested',
            mode='lines',
            line=dict(color='#FF6B6B', width=2),
            hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Invested: €%{y:,.2f}<extra></extra>'
        )
    )

    # Calculate y-axis range with ±10% margin
    all_values = pd.concat([portfolio_df['total_value'], portfolio_df['cumulative_invested']])
    data_min = all_values.min()
    data_max = all_values.max()
    data_range = data_max - data_min
    margin = data_range * 0.15
    
    y_min = data_min - margin
    y_max = data_max + margin

    # Layout with dark theme
    fig.update_layout(
        title='DEGIRO Portfolio Progression',
        xaxis_title='Date',
        yaxis_title='Value (EUR)',
        yaxis=dict(range=[y_min, y_max]),
        hovermode='x unified',
        height=500,
        template='plotly_dark',
        font=dict(size=11),
        plot_bgcolor='#111111',
        paper_bgcolor='#1a1a1a',
        legend=dict(
            x=0.02,
            y=0.98,
            bgcolor='rgba(0, 0, 0, 0.6)',
            bordercolor='#444444',
            borderwidth=1
        )
    )

    fig.show()
    print("✓ Visualization complete")
else:
    print("✗ No data to visualize")

✓ Visualization complete
